# Step 2: Exploratory Data Analysis (EDA)

Purpose: Assess data quality, benchmark model performance, and plan feature engineering for the Laser Resonator Beam Alignment Recommendation System.

Primary outputs:
- `ai/memory/eda_insights.md`: Memory report documenting all findings
- `ai/implementation/step_02/outcome.md`: Step summary with recommendations
- Promoted EDA helper functions to `src/` if any reusable logic is identified.

## 0. Notebook Setup and Paths

All paths are resolved relative to the project root so the notebook can be run from the repository or from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root by walking upward to `pyproject.toml`."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SRC_DIR = PROJECT_ROOT / "src"
MEMORY_DIR = PROJECT_ROOT / "ai" / "memory"

DATASET_PATH = PROCESSED_DIR / "dataset_001.csv"

MEMORY_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, DATASET_PATH

## Unit 01: Data Quality Assessment

Load `dataset_001.csv` and perform initial data quality assessment:
- Check for constant/near-constant features
- Check for potential data leakage (features with extreme correlation to targets)
- Analyze feature distributions and detect outliers
- Validate the sentinel imputation strategy for missing values

In [ ]:
# Unit 01: Load dataset_001.csv
df = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
display(df.head(2))

In [ ]:
# Check for constant/near-constant features
constant_features = []
near_constant_features = []

for col in df.columns:
    if df[col].nunique() == 1:
        constant_features.append(col)
    elif df[col].nunique() < 10 and df[col].dtype != 'object':
        near_constant_features.append(col)

print(f"Constant features: {constant_features}")
print(f"Near-constant features: {near_constant_features}")

In [ ]:
# Check for potential data leakage (high correlation with targets)
target_cols = [col for col in df.columns if col.startswith('target_')]
feature_cols = [col for col in df.columns if not col.startswith('target_') and df[col].dtype in ['int64', 'float64']]

# Calculate correlations with targets
correlations = pd.DataFrame()
for feat in feature_cols:
    for target in target_cols:
        corr = df[feat].corr(df[target])
        if abs(corr) > 0.9:  # High correlation threshold
            correlations = pd.concat([correlations, pd.DataFrame({'feature': [feat], 'target': [target], 'correlation': [corr]})])

print("Potential data leakage (correlation > 0.9):")
display(correlations if not correlations.empty else "None found")

In [ ]:
# Analyze feature distributions and detect outliers
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print(f"Numeric columns: {len(numeric_cols)}")

# Display basic stats for numeric columns
display(df[numeric_cols].describe())

## Unit 02: Benchmark Model Iteration (0-3 attempts)

Run quick Random Forest classifiers for all 4 parameters (Iris, Z, Pitch, Yaw) and iterate up to 3 times if accuracy < 70%.

In [ ]:
# Unit 02: Prepare features and targets for benchmark models
# Target columns
target_iris = 'target_iris_changed'
target_z = 'target_z_changed'
target_pitch = 'target_pitch_changed'
target_yaw = 'target_yaw_changed'

# Feature columns (exclude target columns and metadata)
exclude_cols = ['meta_index1', 'meta_index2', 'meta_diff_count', 
                'meta_before_experiment_number', 'meta_after_experiment_number']
feature_cols = [col for col in df.columns if col not in exclude_cols and not col.startswith('target_')]

X = df[feature_cols].copy()
print(f"Features: {len(feature_cols)}")
display(X.head(2))

In [ ]:
# Run benchmark Random Forest for Iris parameter
# Placeholder - implement after deciding on encoding/transformations

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# TODO: Apply encoding/transformations if needed
# X_encoded = ...

y_iris = df[target_iris]
X_train, X_test, y_train, y_test = train_test_split(X, y_iris, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Iris benchmark accuracy: {accuracy:.4f}")

In [ ]:
# Run benchmark Random Forest for Z parameter
y_z = df[target_z]
X_train, X_test, y_train, y_test = train_test_split(X, y_z, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Z benchmark accuracy: {accuracy:.4f}")

In [ ]:
# Run benchmark Random Forest for Pitch parameter
y_pitch = df[target_pitch]
X_train, X_test, y_train, y_test = train_test_split(X, y_pitch, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Pitch benchmark accuracy: {accuracy:.4f}")

In [ ]:
# Run benchmark Random Forest for Yaw parameter
y_yaw = df[target_yaw]
X_train, X_test, y_train, y_test = train_test_split(X, y_yaw, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Yaw benchmark accuracy: {accuracy:.4f}")

### Feature Importance Analysis (if accuracy >= 70%)

Extract feature importances from benchmark models to identify top features for each parameter.

In [ ]:
# Placeholder for feature importance extraction
# This should be implemented after reaching >= 70% accuracy

importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

display(feature_importance_df.head(20))

## Unit 03: Feature Engineering Strategy

Document encoding strategies, transformation strategies, and create a feature engineering recommendations matrix.

### Encoding Strategy

Document encoding strategies for categorical and numeric features:

| Feature | Type | Current Format | Recommended Encoding | Reason |
|---------|------|----------------|---------------------|--------|
| _TBD_ | _TBD_ | _TBD_ | _TBD_ | _TBD_ |

### Transformation Strategy

Document transformation strategies for features:

| Feature | Current Scale | Recommended Transform | Reason |
|---------|---------------|----------------------|--------|
| _TBD_ | _TBD_ | _TBD_ | _TBD_ |

### Feature Engineering Recommendations Matrix

Based on benchmark performance and feature importance results:

| Feature | Importance | Current State | Recommended Action | Justification |
|---------|------------|---------------|-------------------|---------------|
| _TBD_ | _TBD_ | _TBD_ | _TBD_ | _TBD_ |

### Redundancy Cases

Document what NOT to do and why:

| Feature | Action | Reason |
|---------|--------|--------|
| _TBD_ | _TBD_ | _TBD_ |

## Unit 04: Split Strategy Assessment

Analyze experiment number distribution for grouped split validity.

In [ ]:
# Unit 04: Analyze experiment number distribution
before_exp_col = 'meta_before_experiment_number'
after_exp_col = 'meta_after_experiment_number'

print(f"Unique before experiment numbers: {df[before_exp_col].nunique()}")
print(f"Unique after experiment numbers: {df[after_exp_col].nunique()}")

print("\nBefore experiment distribution (top 10):")
display(df[before_exp_col].value_counts().head(10))

print("\nAfter experiment distribution (top 10):")
display(df[after_exp_col].value_counts().head(10))

In [ ]:
# Check if experiments overlap between before and after
before_experiments = set(df[before_exp_col].unique())
after_experiments = set(df[after_exp_col].unique())

overlap = before_experiments & after_experiments
print(f"Experiments in both before and after: {len(overlap)}")
print(f"Experiments only in before: {len(before_experiments - after_experiments)}")
print(f"Experiments only in after: {len(after_experiments - before_experiments)}")

## Unit 05: Documentation & Sign-off

Create memory report, update outcome file, and promote reusable EDA functions to `src/`.

### Memory Report Template

Create `ai/memory/eda_insights.md` with the following structure:

```markdown
# EDA Insights Report

Generated: [DATE]

## Summary

_TBD_

## Key Findings

### Data Quality

- **Constant/Near-Constant Features**: _TBD_
- **Potential Data Leakage**: _TBD_
- **Feature Distributions**: _TBD_
- **Outliers**: _TBD_

### Benchmark Model Performance

| Parameter | Accuracy | Assessment |
|-----------|----------|------------|
| Iris | _TBD_ | _TBD_ |
| Z | _TBD_ | _TBD_ |
| Pitch | _TBD_ | _TBD_ |
| Yaw | _TBD_ | _TBD_ |
| **Average** | _TBD_ | _TBD_ |

### Feature Importance (if accuracy >= 70%)

| Parameter | Top Features | Notes |
|-----------|--------------|-------|
| Iris | _TBD_ | _TBD_ |
| Z | _TBD_ | _TBD_ |
| Pitch | _TBD_ | _TBD_ |
| Yaw | _TBD_ | _TBD_ |

### Feature Engineering Strategy

| Feature | Current State | Recommendation | Justification |
|---------|---------------|----------------|---------------|
| _TBD_ | _TBD_ | _TBD_ | _TBD_ |

### Encoding Strategy

| Feature Type | Current Format | Recommended Encoding | Reason |
|--------------|----------------|---------------------|--------|
| _TBD_ | _TBD_ | _TBD_ | _TBD_ |

### Transformation Strategy

| Feature | Current Scale | Recommended Transform | Reason |
|---------|---------------|----------------------|--------|
| _TBD_ | _TBD_ | _TBD_ | _TBD_ |

### Grouped Split Validity Assessment

| Assessment | Finding | Decision |
|------------|---------|----------|
| Experiment number distribution | _TBD_ | _TBD_ |
| Grouped vs Random split | _TBD_ | _TBD_ |

## Recommendations for Step 3

- **Feature Set**: _TBD_
- **Encoding/Transformations**: _TBD_
- **Split Strategy**: _TBD_ (random or grouped by experiment)
- **Next Steps**: _TBD_
```

In [ ]:
# Unit 05: Promote reusable EDA functions to src/

# If any reusable helper functions are identified during EDA,
# they should be promoted to appropriate modules in src/.

# Example:
# - src/eda/quality_checks.py
# - src/eda/encoding_helpers.py
# - src/eda/transformations.py

# TODO: Promote functions if any reusable logic is identified